In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
import os
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Define the device (GPU if available)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Define paths
train_path = r"D:\deeplearning\vscode deep\Birds_25\train"
val_path = r"D:\deeplearning\vscode deep\Birds_25\val"
test_path = r"D:\deeplearning\vscode deep\Birds_25\test"

# Define data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}


In [2]:
# Load datasets using ImageFolder
train_dataset = datasets.ImageFolder(root=train_path, transform=data_transforms['train'])
val_dataset = datasets.ImageFolder(root=val_path, transform=data_transforms['val'])
test_dataset = datasets.ImageFolder(root=test_path, transform=data_transforms['test'])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

# Load MobileNetV3-Large model pretrained on ImageNet
model = models.mobilenet_v3_large(pretrained=True)

# Modify the final layer to match the number of classes in your dataset
num_ftrs = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_ftrs, len(train_dataset.classes))

# Move the model to the device (GPU or CPU)
model = model.to(device)


c:\Users\bhore\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\bhore\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer: SGD with momentum
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)


In [4]:
# Set the number of epochs
num_epochs = 20  # You can set this to any number you prefer

# Add a variable to store the best validation accuracy
best_val_acc = 0.0

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    # Training phase
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        # Backward pass + optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)

    print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    # Validation phase
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)

    print(f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    # Check if the current epoch's validation accuracy is the best we've seen so far
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save the model
        torch.save(model.state_dict(), f'best_model_epoch_{epoch+1}.pth')
        print(f'Best model saved at epoch {epoch+1} with validation accuracy: {best_val_acc:.4f}')

print('Training complete')


Epoch 1/20
----------
Train Loss: 0.9130 Acc: 0.7860
Val Loss: 0.2014 Acc: 0.9513
Best model saved at epoch 1 with validation accuracy: 0.9513
Epoch 2/20
----------
Train Loss: 0.2032 Acc: 0.9432
Val Loss: 0.0944 Acc: 0.9763
Best model saved at epoch 2 with validation accuracy: 0.9763
Epoch 3/20
----------
Train Loss: 0.1247 Acc: 0.9661
Val Loss: 0.0595 Acc: 0.9867
Best model saved at epoch 3 with validation accuracy: 0.9867
Epoch 4/20
----------
Train Loss: 0.0905 Acc: 0.9754
Val Loss: 0.0404 Acc: 0.9906
Best model saved at epoch 4 with validation accuracy: 0.9906
Epoch 5/20
----------
Train Loss: 0.0651 Acc: 0.9829
Val Loss: 0.0293 Acc: 0.9943
Best model saved at epoch 5 with validation accuracy: 0.9943
Epoch 6/20
----------
Train Loss: 0.0503 Acc: 0.9872
Val Loss: 0.0199 Acc: 0.9960
Best model saved at epoch 6 with validation accuracy: 0.9960
Epoch 7/20
----------
Train Loss: 0.0409 Acc: 0.9898
Val Loss: 0.0144 Acc: 0.9974
Best model saved at epoch 7 with validation accuracy: 0.9974

In [2]:
# Testing phase
model.eval()
test_loss = 0.0
test_corrects = 0
accuracy_list = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * inputs.size(0)
        test_corrects += torch.sum(preds == labels.data)
        
        # Calculate accuracy for each batch
        batch_acc = torch.sum(preds == labels.data).double() / inputs.size(0)
        accuracy_list.append(batch_acc.item())

test_loss = test_loss / len(test_dataset)
test_acc = test_corrects.double() / len(test_dataset)

print(f'Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}')

# Plot accuracy for each batch
plt.plot(accuracy_list, label='Batch Accuracy')
plt.xlabel('Batch Number')
plt.ylabel('Accuracy')
plt.title('Batch Accuracy during Testing')
plt.legend()
plt.show()


NameError: name 'model' is not defined

In [5]:
# Load the saved best model
best_model_path = 'best_model_epoch_19.pth'  # Path to the saved model file
model.load_state_dict(torch.load(best_model_path))
model.to(device)
model.eval()  # Set the model to evaluation mode

# Initialize variables for tracking test loss and accuracy
test_loss = 0.0
test_corrects = 0

# No need to compute gradients during testing
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        # Track test loss and accuracy
        test_loss += loss.item() * inputs.size(0)
        test_corrects += torch.sum(preds == labels.data)

# Calculate average test loss and accuracy
test_loss = test_loss / len(test_dataset)
test_acc = test_corrects.double() / len(test_dataset)

print(f'Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}')


C:\Users\bhore\AppData\Local\Temp\ipykernel_11420\1923857467.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


Test Loss: 0.0490 Acc: 0.9865
